In [1]:

MASK = (1 << 64) - 1

K0 = 0xC3A5C85C97CB3127
K1 = 0xB492B66FBE98F273
K2 = 0x9AE16A3B2F90404F
KMUL = 0x9DDFEA08EB382D69


def u64(x):
    return x & MASK


def fetch64(s, pos):
    return int.from_bytes(s[pos:pos + 8], "little")


def fetch32(s, pos):
    return int.from_bytes(s[pos:pos + 4], "little")


def rotate(x, shift):
    x &= MASK
    return ((x >> shift) | (x << (64 - shift))) & MASK


def shift_mix(x):
    x &= MASK
    return x ^ (x >> 47)


def hash_len16(u, v, mul=KMUL):
    a = u64((u ^ v) * mul)
    a ^= a >> 47
    b = u64((v ^ a) * mul)
    b ^= b >> 47
    b = u64(b * mul)
    return b


# made using AI from here

def hash_len0to16(s):
    length = len(s)

    if length >= 8:
        mul = K2 + length * 2
        a = u64(fetch64(s, 0) + K2)
        b = fetch64(s, length - 8)
        c = u64(rotate(b, 37) * mul + a)
        d = u64((rotate(a, 25) + b) * mul)
        return hash_len16(c, d, mul)

    if length >= 4:
        mul = K2 + length * 2
        a = fetch32(s, 0)
        b = fetch32(s, length - 4)
        return hash_len16(u64(length + (a << 3)),b,mul)

    if length > 0:
        a = s[0]
        b = s[length >> 1]
        c = s[length - 1]

        y = a + (b << 8)
        z = length + (c << 2)

        return u64(shift_mix(u64(y * K2) ^ u64(z * K0)) * K2)

    return K2


def hash_len17to32(s):
    length = len(s)
    mul = K2 + length * 2

    a = u64(fetch64(s, 0) * K1)
    b = fetch64(s, 8)
    c = u64(fetch64(s, length - 8) * mul)
    d = u64(fetch64(s, length - 16) * K2)

    return hash_len16(u64(rotate(u64(a + b), 43) + rotate(c, 30) + d),u64(a + rotate(u64(b + K2), 18) + c),mul)


def hash_len33to64(s):
    length = len(s)
    mul = K2 + length * 2

    a = u64(fetch64(s, 0) * K2)
    b = fetch64(s, 8)
    c = fetch64(s, length - 24)
    d = fetch64(s, length - 32)
    e = u64(fetch64(s, 16) * K2)
    f = u64(fetch64(s, 24) * 9)
    g = fetch64(s, length - 8)
    h = u64(fetch64(s, length - 16) * mul)

    u = u64(rotate(u64(a + g), 43) + u64(rotate(b, 30) + c) * 9)

    v = u64(((a + g) ^ d) + f + 1)
    w = u64(int.from_bytes(u64(u + v).to_bytes(8, "little"), "big") * mul + h)

    x = u64(rotate(u64(e + f), 42) + c)

    y = u64(int.from_bytes(u64(v + w).to_bytes(8, "little"), "big") + g)
    y = u64(y * mul)

    z = u64(e + f + c)

    a = u64(int.from_bytes(u64(u64(x + z) * mul + y).to_bytes(8, "little"),"big") + b)

    b = u64(shift_mix(u64(z + a) * mul + d + h) * mul)

    return u64(b + x)


def weak_hash_len32_with_seeds(s, pos, seed_a, seed_b):
    a = fetch64(s, pos)
    b = fetch64(s, pos + 8)
    c = fetch64(s, pos + 16)
    d = fetch64(s, pos + 24)

    seed_a = u64(seed_a + a)
    seed_b = rotate(u64(seed_b + seed_a + d), 21)

    c1 = seed_a
    seed_a = u64(seed_a + b + c)
    seed_b = u64(seed_b + rotate(seed_a, 44))

    return u64(seed_a + d), u64(seed_b + c1)


def cityhash64(data):
    
    if isinstance(data, str):
        s = data.encode("utf-8")
    else:
        s = bytes(data)

    length = len(s)

    if length <= 16:
        return hash_len0to16(s)

    if length <= 32:
        return hash_len17to32(s)

    if length <= 64:
        return hash_len33to64(s)

    x = fetch64(s, length - 40)
    y = u64(fetch64(s, length - 16) + fetch64(s, length - 56))

    z = hash_len16(u64(fetch64(s, length - 48) + length),fetch64(s, length - 24))

    v = weak_hash_len32_with_seeds(s, length - 64, length, z)

    w = weak_hash_len32_with_seeds(s, length - 32, u64(y + K1), x)

    x = u64(x * K1 + fetch64(s, 0))

  
    remaining = (length - 1) & ~63
    pos = 0

    while remaining:
        x = u64(rotate(u64(x + y + v[0] + fetch64(s, pos + 8)),37) * K1)

        y = u64(rotate(u64(y + v[1] + fetch64(s, pos + 48)),42) * K1)

        x ^= w[1]
        y = u64(y + v[0] + fetch64(s, pos + 40))

        z = u64(rotate(u64(z + w[0]), 33) * K1)

        v = weak_hash_len32_with_seeds(s, pos, u64(v[1] * K1), u64(x + w[0]))

        w = weak_hash_len32_with_seeds(s, pos + 32, u64(z + w[1]),u64(y + fetch64(s, pos + 16)))

        x, z = z, x
        pos += 64
        remaining -= 64

    return hash_len16(u64(hash_len16(v[0], w[0]) + u64(shift_mix(y) * K1) + z),u64(hash_len16(v[1], w[1]) + x))



print(cityhash64(""))
print(cityhash64("hello"))
print(cityhash64("hello world"))
print(cityhash64("a" * 100))


11160318154034397263
13009744463427800296
6381520714923946011
17959351148904136240
